In [ ]:
!pip -q install requests pandas

In [ ]:
import requests
import pandas as pd
import time

BASE_URL = "https://world.openbeautyfacts.org/api/v2/search"

params = {
    "search_terms": "moisturizer",
    "page_size": 100,
    "page": 1,
    "fields": (
        "code,product_name,brands,categories,"
        "ingredients_text,ingredients_tags,"
        "allergens_tags,countries,image_url,"
        "last_modified_t"
    )
}

response = requests.get(
    BASE_URL,
    params=params,
    headers={"User-Agent": "Ingredia/1.0 (student research project)"},
    timeout=30
)

response.raise_for_status()
data = response.json()

products = data.get("products", [])
df = pd.json_normalize(products)

print("Products retrieved:", len(df))
display(df.head())

Products retrieved: 100


,code,countries,last_modified_t,image_url,allergens_tags,brands,categories,ingredients_text,product_name,ingredients_tags
0,0015034519916,Poland,1792870351,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,9441720461824,Morocco,1792870252,https://images.openbeautyfacts.org/images/prod...,NaN,NaN,NaN,NaN,NaN,NaN
2,40899329,"Finland, Ireland, Spain",1792870094,https://images.openbeautyfacts.org/images/prod...,[],Dentalux,,,Dental tape,NaN
3,8720181190407,Italy,1792870057,https://images.openbeautyfacts.org/images/prod...,NaN,NaN,NaN,NaN,Dove sapone mani idratante,NaN
4,3760143830601,France,1792869619,https://images.openbeautyfacts.org/images/prod...,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Keep records with at least a product name or barcode
df = df.dropna(
    subset=["product_name", "code"],
    how="all"
)

# Remove duplicate products using their barcode
if "code" in df.columns:
    df = df.drop_duplicates(subset=["code"])

# Keep records that have an ingredient list
if "ingredients_text" in df.columns:
    df = df[
        df["ingredients_text"].notna()
        & (df["ingredients_text"].str.strip() != "")
    ]

df.to_csv("ingredia_products.csv", index=False)

print("Usable products:", len(df))
print("Saved as ingredia_products.csv")

Usable products: 27
Saved as ingredia_products.csv


In [ ]:
from google.colab import files

files.download("ingredia_products.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Cell 1: Install libraries
!pip -q install requests pandas huggingface_hub

In [ ]:
# Cell 2: Download the INCIDB free sample from its public GitHub repository

import requests
import zipfile
import io
import os

repo = "INCIDB/cosmetics-skincare-inci-database"
file_path = "samples/incidb_free_samples.zip"

api_url = f"https://api.github.com/repos/{repo}/contents/{file_path}"

r = requests.get(api_url, timeout=30)
r.raise_for_status()
file_info = r.json()

download_url = file_info.get("download_url")
if not download_url:
    raise RuntimeError(
        "Sample ZIP download URL was not returned. "
        "Open the INCIDB repository and check its samples folder."
    )

zip_response = requests.get(download_url, timeout=120)
zip_response.raise_for_status()

os.makedirs("/content/ingredia_data/incidb", exist_ok=True)

with zipfile.ZipFile(io.BytesIO(zip_response.content)) as z:
    z.extractall("/content/ingredia_data/incidb")
    print("Files in INCIDB sample:")
    for name in z.namelist():
        print(name)

print("INCIDB sample downloaded and extracted.")

Files in INCIDB sample:
brands.csv
brands.parquet
products.csv
products.parquet
ingredients.csv
ingredients.parquet
product_ingredients.csv
product_ingredients.parquet
ingredient_name_map.csv
ingredient_name_map.parquet
fragrance_allergens.csv
fragrance_allergens.parquet
regulatory_status.csv
regulatory_status.parquet
INCIDB sample downloaded and extracted.


In [ ]:
import pandas as pd
from pathlib import Path

INCIDB_DIR = Path("/content/ingredia_data/incidb")

incidb_tables = {}

for path in INCIDB_DIR.rglob("*.csv"):
    try:
        table = pd.read_csv(
            path,
            sep="|",
            dtype=str,
            keep_default_na=False,
            on_bad_lines="warn"
        )
        table_name = path.stem.lower()
        incidb_tables[table_name] = table

        print(f"\n{table_name}: {table.shape}")
        display(table.head(3))

    except Exception as e:
        print(f"Could not read {path.name}: {e}")

print("\nLoaded tables:", list(incidb_tables.keys()))


ingredients: (1123, 20)


,ingredient_id,inci_name,cosing_matched,cosing_ref_no,cas_number,ec_number,functions,chemical_description,cosing_restriction,cosing_update_date,annex_ii,annex_iii,annex_iv,annex_v,annex_vi,is_common_allergen,allergen_source,comedogenic_rating,is_fungal_acne_trigger,rating_source
0,51,( /-),0,,,,,,,,,,,,,0,,,,
1,203,(F.I.L B38252/3),0,,,,,,,,,,,,,0,,,,
2,204,(F.I.L B48924/1),0,,,,,,,,,,,,,0,,,,



brands: (152, 2)


,brand_id,name
0,8,Dove
1,10,The Ordinary
2,12,Head & Shoulders



product_ingredients: (4089, 4)


...


products: (200, 7)


,product_id,brand_id,barcode_ean,name,obf_categories_tags,raw_ingredient_text,created_at
0,498,467,3014284027445,Fixation extra forte,fr:cremes-adhesives-pour-protheses-dentaires,"Calcium/Zinc PVM/MA Copolymer, Paraffinum liqu...",2026-10-10 10:34:33
1,580,529,3140100336979,Shampooing expert anti pelliculaire Citrus,en:hair;en:shampoos;en:anti-dandruff-shampoos,"AQUA/WATER/EAU, SODIUM LAURETH SULFATE, SODIUM...",2026-10-10 10:34:34
2,646,583,3178041308939,Kératine Color 4.6 Châtain Chocolat,en:hair;en:hair-dyes,"Crème Colorante : Aqua, Propylene Glycol, Cety...",2026-10-10 10:34:34



ingredient_name_map: (1467, 7)


,raw_name,canonical_name,method,confidence,ingredient_id,part_index,split_rule
0,TOCOPHEROL. MAY CONTAIN (+/-): CI 77891,( /-),unresolved_residual,0.0,51,2,colon+period
1,PARFUM / FRAGRANCE. (F.I.L B38252/3),(F.I.L B38252/3),unresolved_residual,0.0,203,2,period
2,PARFUM / FRAGRANCE. (F.I.L B48924/1),(F.I.L B48924/1),unresolved_residual,0.0,204,2,period



Loaded tables: ['ingredients', 'brands', 'product_ingredients', 'fragrance_allergens', 'regulatory_status', 'products', 'ingredient_name_map']


In [ ]:
# Cell 4: Fetch product records from Open Beauty Facts

import requests
import pandas as pd
import time
import json
from pathlib import Path

DATA_DIR = Path("/content/ingredia_data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

BASE_URL = "https://world.openbeautyfacts.org/api/v2/search"

search_terms = [
    "moisturizer",
    "cleanser",
    "sunscreen",
    "serum",
    "shampoo"
]

all_products = []

for term in search_terms:
    for page in range(1, 4):  # Up to 3 pages per term
        params = {
            "search_terms": term,
            "page_size": 100,
            "page": page,
            "fields": (
                "code,product_name,brands,categories,"
                "ingredients_text,ingredients_tags,allergens_tags,"
                "countries,image_url,last_modified_t"
            )
        }

        response = requests.get(
            BASE_URL,
            params=params,
            headers={"User-Agent": "Ingredia/1.0 (educational project)"},
            timeout=30
        )
        response.raise_for_status()

        batch = response.json().get("products", [])
        if not batch:
            break

        for product in batch:
            product["search_term"] = term

        all_products.extend(batch)
        print(f"{term}, page {page}: {len(batch)} products")
        time.sleep(1)

products_df = pd.DataFrame(all_products)

if not products_df.empty and "code" in products_df.columns:
    products_df = products_df.drop_duplicates(subset=["code"])

# Convert list/dict values into JSON strings for reliable CSV storage
for col in products_df.columns:
    products_df[col] = products_df[col].apply(
        lambda x: json.dumps(x, ensure_ascii=False)
        if isinstance(x, (list, dict)) else x
    )

products_df.to_csv(DATA_DIR / "open_beauty_facts_products.csv", index=False)

print("Unique products collected:", len(products_df))
display(products_df.head())

moisturizer, page 1: 100 products
moisturizer, page 2: 100 products
moisturizer, page 3: 100 products
cleanser, page 1: 100 products
cleanser, page 2: 100 products
cleanser, page 3: 100 products
sunscreen, page 1: 100 products
sunscreen, page 2: 100 products
sunscreen, page 3: 100 products
serum, page 1: 100 products
serum, page 2: 100 products
serum, page 3: 100 products
shampoo, page 1: 100 products
shampoo, page 2: 100 products
shampoo, page 3: 100 products
Unique products collected: 300


,code,countries,last_modified_t,search_term,image_url,allergens_tags,brands,categories,ingredients_text,product_name,ingredients_tags
0,0015034519916,Poland,1792870351,moisturizer,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,9441720461824,Morocco,1792870252,moisturizer,https://images.openbeautyfacts.org/images/prod...,NaN,NaN,NaN,NaN,NaN,NaN
2,40899329,"Finland, Ireland, Spain",1792870094,moisturizer,https://images.openbeautyfacts.org/images/prod...,[],Dentalux,,,Dental tape,NaN
3,8720181190407,Italy,1792870057,moisturizer,https://images.openbeautyfacts.org/images/prod...,NaN,NaN,NaN,NaN,Dove sapone mani idratante,NaN
4,3760143830601,France,1792869619,moisturizer,https://images.openbeautyfacts.org/images/prod...,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Cell 5: Download the CureSkin CSV from Hugging Face

from huggingface_hub import hf_hub_download
from pathlib import Path
import shutil
import pandas as pd

DATA_DIR = Path("/content/ingredia_data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

source_path = hf_hub_download(
    repo_id="cureskin/indian-skincare-ingredient-flags",
    filename="ingredient-flags.csv",
    repo_type="dataset"
)

destination = DATA_DIR / "cureskin_ingredient_flags.csv"
shutil.copy2(source_path, destination)

cureskin_df = pd.read_csv(destination)

print("Rows:", len(cureskin_df))
print("Columns:", cureskin_df.columns.tolist())
display(cureskin_df.head())

ingredient-flags.csv:   0%|          | 0.00/145k [00:00<?, ?B/s]

Rows: 1448
Columns: ['inci_name', 'display_name', 'function', 'is_active', 'fa_trigger', 'comedogenic', 'fragrance', 'eu_allergen', 'alcohol_type', 'pregnancy_caution', 'product_count']


,inci_name,display_name,function,is_active,fa_trigger,comedogenic,fragrance,eu_allergen,alcohol_type,pregnancy_caution,product_count
0,aqua,Aqua,Solvent / vehicle,0,0.0,NaN,0,0,NaN,0,1204
1,niacinamide,Niacinamide,Cell-communicating / barrier-repair active,1,0.0,0.0,0,0,NaN,0,1153
2,glycerin,Glycerin,Humectant,0,0.0,0.0,0,0,NaN,0,2141
3,hyaluronic acid,Hyaluronic Acid,Humectant / hydrating agent,1,0.0,0.0,0,0,NaN,0,346
4,panthenol,Panthenol,Humectant / skin-conditioning agent,0,0.0,NaN,0,0,NaN,0,698


In [ ]:
# Cell 6: Download California's Chemicals in Cosmetics CSV

import requests
import pandas as pd
from pathlib import Path

DATA_DIR = Path("/content/ingredia_data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

metadata_url = (
    "https://publish.data.ca.gov/api/3/action/package_show"
    "?id=chemicals-in-cosmetics"
)

meta_response = requests.get(metadata_url, timeout=30)
meta_response.raise_for_status()
metadata = meta_response.json()

if not metadata.get("success"):
    raise RuntimeError(
        "Could not retrieve dataset metadata. "
        "Download the CSV manually from the official page."
    )

resources = metadata["result"]["resources"]

csv_resources = [
    item for item in resources
    if item.get("format", "").lower() == "csv"
]

if not csv_resources:
    raise RuntimeError(
        "No CSV resource found. Use the official dataset page below."
    )

csv_url = csv_resources[0]["url"]
print("CSV source:", csv_url)

csv_response = requests.get(csv_url, timeout=120)
csv_response.raise_for_status()

csv_path = DATA_DIR / "california_chemicals_in_cosmetics.csv"
csv_path.write_bytes(csv_response.content)

california_df = pd.read_csv(csv_path, low_memory=False)

print("Rows:", len(california_df))
print("Columns:", california_df.columns.tolist())
display(california_df.head())

CSV source: https://data.chhs.ca.gov/dataset/596b5eed-31de-4fd8-a645-249f3f9b19c4/resource/57da6c9a-41a7-44b0-ab8d-815ff2cd5913/download/cscpopendata.csv
Rows: 114635
Columns: ['CDPHId', 'ProductName', 'CSFId', 'CSF', 'CompanyId', 'CompanyName', 'BrandName', 'PrimaryCategoryId', 'PrimaryCategory', 'SubCategoryId', 'SubCategory', 'CasId', 'CasNumber', 'ChemicalId', 'ChemicalName', 'InitialDateReported', 'MostRecentDateReported', 'DiscontinuedDate', 'ChemicalCreatedAt', 'ChemicalUpdatedAt', 'ChemicalDateRemoved', 'ChemicalCount']


,CDPHId,ProductName,CSFId,CSF,CompanyId,CompanyName,BrandName,PrimaryCategoryId,PrimaryCategory,SubCategoryId,...,CasNumber,ChemicalId,ChemicalName,InitialDateReported,MostRecentDateReported,DiscontinuedDate,ChemicalCreatedAt,ChemicalUpdatedAt,ChemicalDateRemoved,ChemicalCount
0,2,ULTRA COLOR RICH EXTRA PLUMP LIPSTICK-ALL SHADES,NaN,NaN,4,New Avon LLC,AVON,44,Makeup Products (non-permanent),53,...,13463-67-7,6,Titanium dioxide,06/17/2009,08/28/2013,02/01/2011,07/09/2009,07/09/2009,NaN,1
1,3,Glover's Medicated Shampoo,NaN,NaN,338,J. Strickland & Co.,Glover's,18,Hair Care Products (non-coloring),25,...,65996-92-1,4,Distillates (coal tar),07/01/2009,07/01/2009,NaN,07/01/2009,07/01/2009,NaN,2
2,3,Glover's Medicated Shampoo,NaN,NaN,338,J. Strickland & Co.,Glover's,18,Hair Care Products (non-coloring),25,...,140-67-0,5,Estragole,07/01/2009,07/01/2009,NaN,07/02/2009,07/02/2009,NaN,2
3,4,PRECISION GLIMMER EYE LINER-ALL SHADES �,NaN,NaN,4,New Avon LLC,AVON,44,Makeup Products (non-permanent),46,...,13463-67-7,7,Titanium dioxide,07/09/2009,08/28/2013,NaN,07/09/2009,07/09/2009,NaN,1
4,5,AVON BRILLIANT SHINE LIP GLOSS-ALL SHADES �,NaN,NaN,4,New Avon LLC,AVON,44,Makeup Products (non-permanent),52,...,13463-67-7,8,Titanium dioxide,07/09/2009,08/28/2013,02/01/2011,07/09/2009,07/09/2009,NaN,1


In [ ]:
# Cell 7: Search PubMed for ingredient-related research

import requests
import pandas as pd
import xml.etree.ElementTree as ET
from pathlib import Path

DATA_DIR = Path("/content/ingredia_data")
BASE = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"

query = (
    '("cosmetic ingredients"[Title/Abstract] OR '
    '"skin irritation"[Title/Abstract] OR '
    '"contact dermatitis"[Title/Abstract])'
)

# Step 1: Search for article IDs
search_response = requests.get(
    BASE + "esearch.fcgi",
    params={
        "db": "pubmed",
        "term": query,
        "retmax": 100,
        "retmode": "json",
        "sort": "relevance"
    },
    timeout=30
)
search_response.raise_for_status()

pmids = search_response.json()["esearchresult"]["idlist"]
print("Article IDs found:", len(pmids))

# Step 2: Retrieve article metadata and abstracts
articles = []

if pmids:
    fetch_response = requests.get(
        BASE + "efetch.fcgi",
        params={
            "db": "pubmed",
            "id": ",".join(pmids),
            "rettype": "abstract",
            "retmode": "xml"
        },
        timeout=60
    )
    fetch_response.raise_for_status()

    root = ET.fromstring(fetch_response.content)

    for article in root.findall(".//PubmedArticle"):
        pmid = article.findtext(".//PMID", default="")
        title = " ".join(
            article.findtext(".//ArticleTitle", default="").split()
        )

        abstract_parts = [
            " ".join(node.itertext()).strip()
            for node in article.findall(".//Abstract/AbstractText")
        ]

        articles.append({
            "pmid": pmid,
            "title": title,
            "abstract": " ".join(abstract_parts),
            "url": f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/"
        })

pubmed_df = pd.DataFrame(articles)
pubmed_df.to_csv(DATA_DIR / "pubmed_ingredient_research.csv", index=False)

display(pubmed_df.head())

Article IDs found: 100


,pmid,title,abstract,url
0,31757238,Allergic Contact Dermatitis.,"Allergic contact dermatitis is common, resulti...",https://pubmed.ncbi.nlm.nih.gov/31757238/
1,35433115,Irritant Contact Dermatitis - a Review.,Irritant contact dermatitis is the most common...,https://pubmed.ncbi.nlm.nih.gov/35433115/
2,37061220,TFOS Lifestyle: Impact of cosmetics on the ocu...,In this report the use of eye cosmetic product...,https://pubmed.ncbi.nlm.nih.gov/37061220/
3,29122160,Contact Dermatitis.,,https://pubmed.ncbi.nlm.nih.gov/29122160/
4,30466685,Contact Dermatitis: Emerging Trends.,The field of contact dermatitis is constantly ...,https://pubmed.ncbi.nlm.nih.gov/30466685/


In [ ]:
# Cell 8: Download the Kaggle dataset
# Kaggle authentication may be required.

!pip -q install kagglehub

import kagglehub
import os

kaggle_path = kagglehub.dataset_download(
    "eward96/skincare-products-clean-dataset"
)

print("Dataset downloaded to:", kaggle_path)

for root, dirs, files in os.walk(kaggle_path):
    for filename in files:
        print(os.path.join(root, filename))

Using Colab cache for faster access to the 'skincare-products-clean-dataset' dataset.
Dataset downloaded to: /kaggle/input/skincare-products-clean-dataset
/kaggle/input/skincare-products-clean-dataset/skincare_products_clean.csv


In [ ]:
import shutil
from google.colab import files

# Zip the entire ingredia_data directory
shutil.make_archive("/content/ingredia_data_all", "zip", "/content/ingredia_data")

print("Ready to download files...")

# Download the zipped datasets package
files.download("/content/ingredia_data_all.zip")

# Download the main ingredia_products.csv file
files.download("/content/ingredia_products.csv")

Ready to download files...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import drive
import shutil
from pathlib import Path

# Mount Google Drive
drive.mount('/content/drive')

# Define paths
src_dir = Path('/content/ingredia_data')
src_csv = Path('/content/ingredia_products.csv')
src_zip = Path('/content/ingredia_data_all.zip')

# Target directory in Google Drive
dest_dir = Path('/content/drive/MyDrive/ingredia_backup')
dest_dir.mkdir(parents=True, exist_ok=True)

print("Copying files to Google Drive...")

# Copy the individual files directory
if src_dir.exists():
    shutil.copytree(src_dir, dest_dir / 'ingredia_data', dirs_exist_ok=True)
    print("Copied 'ingredia_data' directory.")

# Copy the main CSV file
if src_csv.exists():
    shutil.copy2(src_csv, dest_dir / 'ingredia_products.csv')
    print("Copied 'ingredia_products.csv'.")

# Copy the ZIP archive
if src_zip.exists():
    shutil.copy2(src_zip, dest_dir / 'ingredia_data_all.zip')
    print("Copied 'ingredia_data_all.zip'.")

print(f"All files successfully copied to Google Drive at: {dest_dir}")

Mounted at /content/drive
Copying files to Google Drive...
Copied 'ingredia_data' directory.
Copied 'ingredia_products.csv'.
Copied 'ingredia_data_all.zip'.
All files successfully copied to Google Drive at: /content/drive/MyDrive/ingredia_backup
